# Task B: NYC Trip Duration Prediction — the Same Ladder at Scale

**Secondary task of the v3 modelling layer.** Predicts `trip_duration_seconds`
using only information available **at pickup time** — pickup hour, pickup
day of week, pickup zone, passenger count, trip distance — from `fact_trip`,
the platform's 40,421,155-row quality-gated NYC yellow-taxi corpus for 2024
(`docs/data_dictionary.md`). This notebook exists to show the same
feature-engineering ladder from `01_nigeria_petrol_forecasting.ipynb`
**generalises** to a completely different scale — tens of millions of rows
instead of hundreds — using the identical shared code in
`src/modelling/{splits,features,encoders,ladder,repeats}.py`.

Trip distance is knowable before a trip starts because a passenger states a
destination before departure, and its use here as a pickup-time feature is
standard practice in published TLC duration-prediction work. Dropoff time
and every other post-trip field are never used.

**The model is fixed across every rung, under two hyperparameter variants**
(`original (untuned, 300 trees)` and `capacity-controlled (CV-selected on
V0)`, §5b), exactly as in Task A.

## What changed in this version, and why it mattered

Two problems made this notebook's earlier headline — roughly a 3.4%
improvement in MAE from the baseline to the best rung — impossible to
defend, and both are fixed here.

**1. The sample was not reproducible.** Rows were drawn with DuckDB's
`USING SAMPLE n ROWS (reservoir, 796)`. That seeds the sampler, but
reservoir sampling's output also depends on the order rows arrive, and a
`read_parquet('*.parquet')` glob does not guarantee a stable scan order
between container runs. Two nominally identical executions drew different
rows and reported different numbers. Sampling is now **content-addressed**:
each trip is assigned to a fixed bucket by the leading hex digits of its own
`trip_id` — itself a deterministic MD5 over the trip's business attributes —
so the selected rows depend on nothing but the data
(`src/modelling/splits.trip_bucket_predicate`,
`tests/test_sampling_determinism.py`, which proves the selection survives a
deliberate reordering and is identical across separate OS processes).

**2. A 3.4% effect was never compared against the variation it had to beat.**
Every rung now runs **five times** (seeds `1..5`), varying the model's
`random_state` **and** the sample bucket together, so a repeat reflects both
the model's randomness and the luck of which rows were drawn. Results are
reported as a mean with a standard deviation and a range, and rung-to-rung
differences are compared **within seed** and reported with the number of
repeats that agree on the direction (§9-10). A difference that does not hold
its sign across all five repeats is not reported as an established effect.

**Two reference predictors (§6)** anchor the absolute level: the training
mean, and distance over the fleet's average speed. The ladder is measured
against those, not against any published figure — a published NYC
trip-duration error is not comparable to this one unless the window,
filtering, sampling and target definition all match, and they do not.

In [1]:
import gc
import time
import warnings
import sys
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=UserWarning)

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import settings
from src.viz import style as vizstyle
from src.utils.db import connect
from src.modelling.splits import (
    nyc_time_split, assert_split_is_time_ordered,
    NYC_TRAIN_MONTHS, NYC_TEST_MONTHS, NYC_TRAIN_SAMPLE_SIZE, NYC_TEST_SAMPLE_SIZE,
    NYC_BUCKET_COUNT, REPEAT_SEEDS, bucket_for_seed, trip_bucket_predicate,
    RAW_TRIP_ID_SQL,
)
from src.modelling.features import add_leaky_zone_hour_avg_nyc, add_pit_zone_hour_avg_nyc
from src.modelling.encoders import one_hot_encode, target_encode_pit
from src.modelling.ladder import (
    run_rung, score_predictions, LGBM_PARAMS,
    VARIANT_ORIGINAL, VARIANT_CAPACITY_CONTROLLED, VARIANT_REFERENCE,
)
from src.modelling.tuning import expanding_window_folds, select_hyperparameters_cv
from src.modelling.baselines import predict_train_mean, predict_distance_over_mean_speed
from src.modelling.repeats import (
    Pair, adjacent_pairs, aggregate_repeats, paired_comparisons, best_rung_by_mean_mae,
    results_to_long_frame,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

NOTEBOOK_STARTED = time.perf_counter()
TARGET = "trip_duration_seconds"

print(f"train months: {NYC_TRAIN_MONTHS[0]}-{NYC_TRAIN_MONTHS[-1]} 2024, "
      f"test months: {NYC_TEST_MONTHS} 2024")
print(f"repeat seeds: {REPEAT_SEEDS}  ->  buckets {[bucket_for_seed(s) for s in REPEAT_SEEDS]} "
      f"of {NYC_BUCKET_COUNT}")
print()
print("ORIGINAL hyperparameters (first ladder variant, every rung):")
LGBM_PARAMS

train months: 1-10 2024, test months: (11, 12) 2024
repeat seeds: (1, 2, 3, 4, 5)  ->  buckets [0, 1, 2, 3, 4] of 17

ORIGINAL hyperparameters (first ladder variant, every rung):


{'n_estimators': 300,
 'learning_rate': 0.05,
 'num_leaves': 31,
 'max_depth': -1,
 'min_child_samples': 20,
 'subsample': 0.8,
 'subsample_freq': 1,
 'colsample_bytree': 0.8,
 'random_state': 796,
 'n_jobs': 4,
 'verbosity': -1}

## 1. Deterministic, content-addressed sampling

Training eight rungs, under two hyperparameter variants, five times each, on
all 40.4M rows is unnecessary: this notebook compares rungs *relative to each
other*. Each repeat therefore works on **one bucket** of the corpus.

A trip's bucket is `('0x' || substr(trip_id, 1, 8))::UBIGINT % 17`.
`trip_id` is already a deterministic MD5 over the trip's business content
(`sql/ddl/silver_nyc_trip.sql`), so its leading hex digits are a uniformly
distributed number that belongs to the row itself and never changes. Selecting
one bucket therefore returns **identical rows on every run**, on any machine,
regardless of scan order — the property the previous reservoir draw lacked.

**17 buckets** lands each bucket near the sizes the old draw targeted
(~1,956,000 training and ~421,000 test rows against targets of 2,000,000 and
400,000). The **realised counts are printed below rather than assumed**: a
bucket's size is a property of the data, not a number this notebook chooses.

The raw TLC Parquet that rung V0 reads has no `trip_id` — the quality gate
assigns it — so V0 recomputes the identical MD5 expression inline from the
same raw columns (`splits.RAW_TRIP_ID_SQL`). Same rule, same determinism, on
a source that has not been through the gate.

**Both pulls also carry an explicit `ORDER BY`, and that is load-bearing
rather than tidiness.** Selecting a deterministic *set* of rows is not
sufficient for a reproducible model: DuckDB's parallel scan returns rows in
no guaranteed order, and gradient boosting is order-sensitive — bagging
selects by position and histogram split ties break by order. Measured
directly on this data: without `ORDER BY`, two processes given provably
identical rows (same row count, same checksum of the target, same sum of
distances) fit models whose test predictions summed to 202,240,914 and
203,933,890. With `ORDER BY trip_id`, both processes produced exactly
208,095,622.694010. An earlier version of this notebook omitted the ordering
and consequently did not reproduce between runs, which moved one
rung-to-rung conclusion; `docs/methodology_notes.md` §12 records it.

In [2]:
RAW_GLOB = str(settings.RAW_NYC / "*.parquet")

def pull_raw(con, months, bucket):
    '''V0's un-gated draw, straight from the TLC Parquet files.'''
    month_list = ", ".join(str(m) for m in months)
    query = f'''
        SELECT
            PULocationID                                              AS pickup_zone_raw,
            EXTRACT(hour  FROM tpep_pickup_datetime)::TINYINT          AS pickup_hour,
            date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime)
                                                                        AS trip_duration_seconds,
            trip_distance                                             AS trip_distance_miles,
            passenger_count
        FROM read_parquet('{RAW_GLOB}')
        WHERE EXTRACT(year FROM tpep_pickup_datetime) = 2024
          AND EXTRACT(month FROM tpep_pickup_datetime) IN ({month_list})
          AND {trip_bucket_predicate(bucket, RAW_TRIP_ID_SQL)}
        ORDER BY {RAW_TRIP_ID_SQL}
    '''
    return con.execute(query).df()


def pull_gated(con, months, bucket):
    '''Every rung from V1 onward: the gated fact, with dimensional context.'''
    month_list = ", ".join(str(m) for m in months)
    query = f'''
        SELECT
            t.pickup_geo_key,
            t.pickup_hour,
            t.trip_distance_miles,
            t.passenger_count,
            t.trip_duration_seconds,
            g.parent_geo_name  AS borough,
            g.region_group     AS service_zone,
            d.day_of_week,
            d.is_weekend,
            d.year,
            d.month_number     AS month,
            w.precipitation_mm,
            w.temp_max_c
        FROM fact_trip t
        JOIN dim_geography g ON g.geo_key = t.pickup_geo_key
        JOIN dim_date d      ON d.date_key = t.pickup_date_key
        LEFT JOIN fact_weather_daily w ON w.date_key = t.pickup_date_key
        WHERE d.year = 2024 AND d.month_number IN ({month_list})
          AND {trip_bucket_predicate(bucket, "t.trip_id")}
        ORDER BY t.trip_id
    '''
    return con.execute(query).df()

print("V0 bucket predicate (raw source), abbreviated:")
print("   ", trip_bucket_predicate(0, "md5(...)"))
print("V1+ bucket predicate (gated source):")
print("   ", trip_bucket_predicate(0, "t.trip_id"))

V0 bucket predicate (raw source), abbreviated:
    ('0x' || substr(md5(...), 1, 8))::UBIGINT % 17 = 0
V1+ bucket predicate (gated source):
    ('0x' || substr(t.trip_id, 1, 8))::UBIGINT % 17 = 0


## 2. Loading one repeat's data

Everything a single repeat needs, in one function: the un-gated draw for V0,
the gated draw with dimensional context for every later rung, the engineered
features, and the time split. It is a function rather than a sequence of
cells because it runs five times, and because a repeat's multi-million-row
intermediates must be freed before the next repeat allocates its own — this
notebook has a ~7.7GB container ceiling and previously hit it.

The **train/test split is by time, never shuffled**: train = January-October
2024, test = November-December 2024, applied to each row's own pickup month
(`splits.nyc_time_split`, with `assert_split_is_time_ordered` run live on
every repeat, not just once).

The leaky and point-in-time features are computed on the train and test rows
**together**, deliberately: the leaky builder's whole failure mode requires
the test rows to be physically present, and the point-in-time builder is safe
computed the same way because every test row's "prior" window is by
construction inside the training period.

In [3]:
class RepeatData:
    '''One repeat's raw draw, gated draw, split and one-hot matrices.'''
    def __init__(self, seed):
        self.seed = seed
        self.bucket = bucket_for_seed(seed)
        t0 = time.perf_counter()

        # A 2GB ceiling, not the pipeline's 6GB default: these are small,
        # bucketed reads, and the rest of the container is needed by pandas
        # and LightGBM. The connection is closed as soon as pulling is done.
        con = connect(settings.WAREHOUSE_DB, read_only=True, memory_limit="2GB", threads=4)
        self.raw_train = pull_raw(con, NYC_TRAIN_MONTHS, self.bucket)
        self.raw_test = pull_raw(con, NYC_TEST_MONTHS, self.bucket)
        gated_train = pull_gated(con, NYC_TRAIN_MONTHS, self.bucket)
        gated_test = pull_gated(con, NYC_TEST_MONTHS, self.bucket)
        con.close()

        modeling = pd.concat([gated_train, gated_test], ignore_index=True)
        del gated_train, gated_test
        gc.collect()

        for col in ("borough", "service_zone"):
            modeling[col] = modeling[col].astype("category")
        # float64 -> float32 halves these columns' footprint with no
        # meaningful precision loss for a duration-in-seconds target, and
        # buys headroom for the V4a one-hot expansion, the peak-memory rung.
        for col in ("trip_distance_miles", "precipitation_mm", "temp_max_c"):
            modeling[col] = modeling[col].astype("float32")

        modeling = add_leaky_zone_hour_avg_nyc(modeling)
        modeling = add_pit_zone_hour_avg_nyc(modeling)
        modeling = target_encode_pit(
            modeling, cat_col="pickup_geo_key", target_col=TARGET,
            time_col="month", smoothing=50.0, out_col="zone_target_enc",
        )

        self.split = nyc_time_split(modeling)
        assert_split_is_time_ordered(self.split, time_col="month")
        del modeling
        gc.collect()

        self.y_train = self.split.train[TARGET]
        self.y_test = self.split.test[TARGET]
        self.seconds = time.perf_counter() - t0

    def build_one_hot(self, common_cols):
        return one_hot_encode(
            self.split.train[common_cols + ["pickup_geo_key"]],
            self.split.test[common_cols + ["pickup_geo_key"]],
            col="pickup_geo_key",
        )

    def describe(self):
        return (f"seed {self.seed} (bucket {self.bucket}): "
                f"raw {len(self.raw_train):,}/{len(self.raw_test):,}, "
                f"gated {len(self.split.train):,}/{len(self.split.test):,} "
                f"train/test rows, loaded in {self.seconds:.1f}s")

first = RepeatData(REPEAT_SEEDS[0])
print(first.describe())
print()
print(f"realised gated training rows: {len(first.split.train):,} "
      f"(the superseded reservoir draw targeted {NYC_TRAIN_SAMPLE_SIZE:,})")
print(f"realised gated test rows:     {len(first.split.test):,} "
      f"(target was {NYC_TEST_SAMPLE_SIZE:,})")
print()
print(first.split.boundary_description)

seed 1 (bucket 0): raw 1,990,219/429,258, gated 1,955,364/420,169 train/test rows, loaded in 132.5s

realised gated training rows: 1,955,364 (the superseded reservoir draw targeted 2,000,000)
realised gated test rows:     420,169 (target was 400,000)

train: 2024 months 1-10 (1955364 rows); test: 2024 months (11, 12) (420169 rows)


In [4]:
first.split.train[[
    "pickup_geo_key", "pickup_hour", "month", TARGET,
    "zone_hour_avg_duration_leaky", "zone_hour_avg_duration_pit", "zone_target_enc",
]].head(8)

,pickup_geo_key,pickup_hour,month,trip_duration_seconds,zone_hour_avg_duration_leaky,zone_hour_avg_duration_pit,zone_target_enc
0,310,7,3,389,609.230462,579.436916,691.106619
1,236,18,5,159,959.712858,884.374850,888.706399
2,228,16,6,1694,1090.098906,1061.399700,902.391446
3,200,16,4,1681,2319.785769,1963.006346,1712.849042
4,227,15,6,559,1042.135279,1031.524646,908.176442
5,226,11,8,1537,981.610419,977.526931,934.959562
6,194,23,5,1605,2060.373964,2002.966698,2426.733762
7,311,11,9,265,779.952716,770.331423,752.848667


## 5b. Once-only hyperparameter selection

The same procedure as Task A, on this task's own data: expanding-window
cross-validation confined to the training months, 8 candidate hyperparameter
sets, `n_estimators` chosen by early stopping, the winner frozen and reused
unchanged at every rung and every repeat. Never re-tuned per rung, and **not
re-run per repeat** — re-selecting per seed would make the capacity-controlled
variant a different model on each repeat, and the spread across repeats would
then confound model randomness with hyperparameter churn.

**One adaptation, stated plainly.** Task B's V0 is deliberately sourced from
the raw, un-gated Parquet and carries no `month` column, which the folds need
to stay time-ordered. The search therefore uses `FEATURES_V1` — the ladder's
own bare, no-engineering starting point once the gate and `month` grain
exist — as its baseline feature set. The selected hyperparameters are still
applied to V0's own raw-sourced features when V0 is fit. It runs on the
already-loaded first repeat's training rows, not on the full corpus, to keep
the search's runtime reasonable.

In [5]:
FEATURES_V1 = ["pickup_hour", "pickup_geo_key", "trip_distance_miles", "passenger_count"]

train_months = sorted(first.split.train["month"].unique())
cv_folds = expanding_window_folds(train_months, min_train_months=6, val_block=1)
print(f"{len(cv_folds)} expanding-window folds over the {len(train_months)} training months:")
for f in cv_folds:
    print(f"  train <= {f.train_months[-1]} ({len(f.train_months)} months)  "
          f"-> validate {f.val_months}")

tuning = select_hyperparameters_cv(
    first.split.train, feature_cols=FEATURES_V1, target_col=TARGET,
    month_col="month", folds=cv_folds,
)
print(f"\nCV search: {len(tuning.cv_table)} candidates x up to {tuning.n_folds} folds "
      f"in {tuning.seconds:.1f}s, on {len(first.split.train):,} training rows")
tuning.cv_table

4 expanding-window folds over the 10 training months:
  train <= 6 (6 months)  -> validate (np.int8(7),)
  train <= 7 (7 months)  -> validate (np.int8(8),)
  train <= 8 (8 months)  -> validate (np.int8(9),)
  train <= 9 (9 months)  -> validate (np.int8(10),)



CV search: 8 candidates x up to 4 folds in 950.8s, on 1,955,364 training rows


,num_leaves,learning_rate,min_child_samples,reg_alpha,reg_lambda,mean_cv_mae,n_folds_used,median_best_iteration,largest_fold_best_iteration
0,31,0.10,50,1.0,1.0,310.851588,4,155,182
1,15,0.10,30,0.5,0.5,311.552999,4,233,254
2,31,0.05,20,1.0,1.0,311.914271,4,240,238
3,7,0.10,20,1.0,1.0,312.406054,4,401,567
4,15,0.05,20,1.0,1.0,312.495165,4,343,440
5,15,0.05,10,0.0,0.0,313.077387,4,320,407
6,7,0.10,5,0.0,0.0,313.486515,4,321,298
7,7,0.03,5,0.0,0.0,313.844476,4,1000,1000


In [6]:
CAPACITY_CONTROLLED_PARAMS = {
    **tuning.best_params,
    "n_estimators": tuning.frozen_n_estimators,
    "max_depth": -1,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "random_state": 796,   # overridden per repeat by run_rung(seed=...)
    "n_jobs": 4,
    "verbosity": -1,
}
print("Selected once, then frozen for every rung and every repeat below:")
CAPACITY_CONTROLLED_PARAMS

Selected once, then frozen for every rung and every repeat below:


{'num_leaves': 31,
 'learning_rate': 0.1,
 'min_child_samples': 50,
 'reg_alpha': 1.0,
 'reg_lambda': 1.0,
 'n_estimators': 182,
 'max_depth': -1,
 'subsample': 0.8,
 'subsample_freq': 1,
 'colsample_bytree': 0.8,
 'random_state': 796,
 'n_jobs': 4,
 'verbosity': -1}

## 6. Reference predictors: is any of this error actually good?

Two predictors that fit no model (`src/modelling/baselines.py`), scored by
the same code as every rung:

- **REF_mean** — predict the training set's mean trip duration for every
  test row. The floor: a rung that cannot beat this has learned nothing.
- **REF_heuristic** — **distance over the fleet's average speed**, the
  obvious thing a practitioner computes with no model at all. Average speed
  is total training distance over total training time, not the mean of
  per-trip speeds, which is dominated by very short trips where a few
  seconds of error implies an absurd speed.

Both are computed on the **gated** test set, so they are directly comparable
to rungs V1 onward and only indicatively comparable to V0, which reads the
un-gated source. Unlike Task A's references these do vary across repeats,
because each repeat draws a different bucket — so their standard deviation
is a genuine measurement of sampling variation, and a useful reading of how
much any rung's number could move for that reason alone.

In [7]:
def reference_results(data, seed):
    ref_mean = score_predictions(
        "REF_mean", "reference: predict the training mean trip duration for every test row",
        data.y_test, predict_train_mean(data.y_train, len(data.y_test)),
        n_train=len(data.y_train),
        notes="no model, no features -- the floor any rung must beat to have learned anything",
        variant=VARIANT_REFERENCE, seed=seed,
    )
    ref_speed = score_predictions(
        "REF_heuristic", "reference: trip distance / the training set's average speed",
        data.y_test, predict_distance_over_mean_speed(data.split.train, data.split.test),
        n_train=len(data.y_train),
        notes="the no-model domain heuristic; average speed is total distance over total time, "
              "not the mean of per-trip speeds",
        variant=VARIANT_REFERENCE, seed=seed,
    )
    return [ref_mean, ref_speed]

for r in reference_results(first, REPEAT_SEEDS[0]):
    print(f"{r.rung:14s} MAE={r.mae:,.1f}s   MAPE={r.mape:.2f}%")

REF_mean       MAE=647.4s   MAPE=211.15%
REF_heuristic  MAE=549.6s   MAPE=47.01%


## 7. The rungs

Each rung declares what it adds. Nothing is fitted until §8, which runs every
rung under both hyperparameter variants, once per repeat — so a rung cannot
accidentally differ between repeats, because there is only one definition of
it.

In [8]:
RUNG_SPECS = []

def add_rung(rung, description, notes, feature_cols=None, kind="columns"):
    RUNG_SPECS.append({"rung": rung, "description": description, "notes": notes,
                       "feature_cols": feature_cols, "kind": kind})
    return feature_cols

### V0 — raw baseline, un-gated source

V0 is built from data that has never passed the quality gate, so V1's
improvement is real rather than nominal (unlike Task A, this source **does**
have meaningful rejected rows: 748,565 of 41,169,720, 1.82% —
`docs/methodology_notes.md` §7). It reads `data/raw/nyc/*.parquet` directly
and derives `pickup_hour` and `trip_duration_seconds` from the raw
timestamps, applying **no reject predicate at all**: negative fares,
non-positive durations and implausible distances all remain, exactly as TLC
published them.

In [9]:
FEATURES_V0 = ["pickup_hour", "pickup_zone_raw", "trip_distance_miles", "passenger_count"]
add_rung("V0", "raw baseline: pickup hour, raw zone id, distance, passenger count "
               "-- un-gated source (negative fares / non-positive durations / implausible "
               "distances all still present)",
         "sourced from data/raw/nyc/*.parquet directly, before the quality gate",
         FEATURES_V0, kind="raw")

['pickup_hour', 'pickup_zone_raw', 'trip_distance_miles', 'passenger_count']

### V1 — quality-gated

Identical shape of features, sourced from `fact_trip` after the gate:
**748,565 of 41,169,720 raw rows (1.82%) excluded** — `negative_money`
(733,787), `non_positive_duration` (13,510), `implausible_distance` (1,038),
`duration_exceeds_ceiling` (230). The raw pickup-zone id is replaced by the
validated `pickup_geo_key` surrogate. `FEATURES_V1` was defined in §5b, where
it also served as the CV search's baseline.

In [10]:
add_rung("V1", "quality-gated: identical shape of features, sourced from fact_trip after the "
               "quality gate (748,565 of 41,169,720 rows rejected, 1.82%)",
         "748,565 rows excluded at the gate (negative_money, non_positive_duration, "
         "implausible_distance, duration_exceeds_ceiling) -- see methodology_notes.md sec 7",
         FEATURES_V1)

['pickup_hour', 'pickup_geo_key', 'trip_distance_miles', 'passenger_count']

### V2 — dimensional features, in place of raw codes

Replaces the raw zone identity with `borough` and `service_zone` (from
`dim_geography`) and adds `day_of_week` / `is_weekend` (from `dim_date`).

In [11]:
FEATURES_V2 = ["pickup_hour", "trip_distance_miles", "passenger_count",
               "borough", "service_zone", "day_of_week", "is_weekend"]
add_rung("V2", "+ dimensional features: borough, service_zone (dim_geography), "
               "day_of_week, is_weekend (dim_date) -- in place of the raw zone code",
         "tests whether conformed-dimension context improves on raw codes, at NYC's scale",
         FEATURES_V2)

['pickup_hour',
 'trip_distance_miles',
 'passenger_count',
 'borough',
 'service_zone',
 'day_of_week',
 'is_weekend']

### V3a — naive historical aggregate (⚠️ LEAKY, DO NOT TRUST THIS NUMBER)

Adds **"this pickup zone and hour's average trip duration,"** computed over
the **entire** sampled dataset — including months after the row being
featurised. A January row can receive a value partly computed from December.
Reported only for comparison against V3b.

In [12]:
FEATURES_V3A = FEATURES_V2 + ["zone_hour_avg_duration_leaky"]
add_rung("V3a", "LEAKY, DO NOT TRUST THIS NUMBER: + zone_hour_avg_duration_leaky "
                "(this zone/hour's average duration over ALL months, including future ones)",
         "LEAKY, DO NOT TRUST THIS NUMBER -- computed with future months present",
         FEATURES_V3A)

['pickup_hour',
 'trip_distance_miles',
 'passenger_count',
 'borough',
 'service_zone',
 'day_of_week',
 'is_weekend',
 'zone_hour_avg_duration_leaky']

### V3b — point-in-time-correct historical aggregate

The same idea, done correctly: the average is computed at **month
granularity**, using only strictly earlier calendar months for that (zone,
hour) cell — the natural refresh cadence of an aggregate a production system
would actually maintain.

In [13]:
FEATURES_V3B = FEATURES_V2 + ["zone_hour_avg_duration_pit"]
add_rung("V3b", "+ zone_hour_avg_duration_pit (zone/hour average duration, strictly-prior "
                "months only)",
         "point-in-time-correct version of the V3a feature -- the trustworthy number",
         FEATURES_V3B)

['pickup_hour',
 'trip_distance_miles',
 'passenger_count',
 'borough',
 'service_zone',
 'day_of_week',
 'is_weekend',
 'zone_hour_avg_duration_pit']

### V4 — encoding comparison: one-hot vs. point-in-time target encoding

The zone categorical (`pickup_geo_key`, up to 265 zones), not used directly
since V1, is reintroduced on top of V3b's feature set via two competing
encodings, mirroring Task A's V4, to engage **Ayinla (2023)** on index-mapped
ordinal encoding at a category count where a sparse expansion is supposed to
start showing its disadvantage.

**An engineering note.** V4a was, through two earlier implementations, the
one rung of either ladder that would not run at all: a dense one-hot
expansion (~2M rows × up to 266 columns, >99% zeros by construction) pushed
this container's memory past its ceiling during LightGBM's own `Dataset`
construction — first via `category_encoders.OneHotEncoder`, then via
`pandas.get_dummies(dtype="int8")`, then via `pandas.get_dummies(sparse=True)`
(which reports a tiny `memory_usage`, but LightGBM's sklearn wrapper silently
densifies a sparse-dtype DataFrame before the booster sees it — the crash was
unchanged). `one_hot_encode` now builds a genuine `scipy.sparse.csr_matrix`
and passes it straight to `LGBMRegressor.fit`; peak memory for this rung fell
from over 7GB to under 1GB. The matrix is built once per repeat and reused
for both hyperparameter variants, since only the model differs between them.

In [14]:
COMMON_V4 = FEATURES_V3B
add_rung("V4a", "+ pickup zone identity, ONE-HOT encoded (scipy.sparse)",
         "one-hot encoding of the up-to-265-zone categorical (scipy.sparse matrix), "
         "fit on train, aligned onto test",
         kind="onehot")

FEATURES_V4B = COMMON_V4 + ["zone_target_enc"]
add_rung("V4b", "+ pickup zone identity, POINT-IN-TIME TARGET encoded (1 numeric column)",
         "smoothed, point-in-time-correct target encoding of the 265-zone categorical "
         "(smoothing=50); engages Ayinla (2023) -- see docs/modelling_notes.md",
         FEATURES_V4B)

['pickup_hour',
 'trip_distance_miles',
 'passenger_count',
 'borough',
 'service_zone',
 'day_of_week',
 'is_weekend',
 'zone_hour_avg_duration_pit',
 'zone_target_enc']

### V5 — external enrichment: weather

Unlike Task A, this rung **is** buildable: `fact_weather_daily` (Open-Meteo's
NYC archive, left-joined on pickup date) supplies precipitation and maximum
temperature. Built on top of V4b.

In [15]:
FEATURES_V5 = FEATURES_V4B + ["precipitation_mm", "temp_max_c"]
add_rung("V5", "+ external enrichment: precipitation_mm, temp_max_c (fact_weather_daily, "
               "joined on pickup date)",
         "fact_weather_daily left-joined on pickup_date_key; reports whether weather "
         "measurably helps predict trip duration",
         FEATURES_V5)

print(f"{len(RUNG_SPECS)} rungs declared: {[s['rung'] for s in RUNG_SPECS]}")

8 rungs declared: ['V0', 'V1', 'V2', 'V3a', 'V3b', 'V4a', 'V4b', 'V5']


## 8. Running the ladder: every rung, both variants, five times

8 rungs × 2 variants × 5 repeats = **80 fits**, plus two reference
predictors per repeat. Each repeat draws its own bucket and uses its own
seed, so the spread across repeats measures sampling variation and model
randomness together — the combined amount a rung's number moves when nothing
meaningful has changed.

Each repeat's data is freed before the next is loaded; the first repeat's
data is already in memory from §2 and is reused rather than re-read.

In [16]:
VARIANTS = ((VARIANT_ORIGINAL, LGBM_PARAMS),
            (VARIANT_CAPACITY_CONTROLLED, CAPACITY_CONTROLLED_PARAMS))

def run_ladder_for_data(data, seed):
    '''Every rung, both hyperparameter variants, on one repeat's data.'''
    out = list(reference_results(data, seed))
    train_oh, test_oh = data.build_one_hot(COMMON_V4)

    for spec in RUNG_SPECS:
        if spec["kind"] == "onehot":
            X_train, X_test, y_tr, y_te = train_oh, test_oh, data.y_train, data.y_test
        elif spec["kind"] == "raw":
            cols = spec["feature_cols"]
            X_train, X_test = data.raw_train[cols], data.raw_test[cols]
            y_tr, y_te = data.raw_train[TARGET], data.raw_test[TARGET]
        else:
            cols = spec["feature_cols"]
            X_train, X_test = data.split.train[cols], data.split.test[cols]
            y_tr, y_te = data.y_train, data.y_test

        for variant, params in VARIANTS:
            out.append(run_rung(
                spec["rung"], spec["description"], X_train, y_tr, X_test, y_te,
                notes=spec["notes"], params=params, variant=variant, seed=seed,
            ))

    del train_oh, test_oh
    gc.collect()
    return out

all_results = []
realised_rows = []   # (seed, bucket, n_train, n_test) -- reported, never assumed
for seed in REPEAT_SEEDS:
    data = first if seed == REPEAT_SEEDS[0] else RepeatData(seed)
    first = None  # release the §2 reference so this repeat's data can be freed
    t0 = time.perf_counter()
    all_results.extend(run_ladder_for_data(data, seed))
    realised_rows.append((seed, data.bucket, len(data.split.train), len(data.split.test)))
    print(f"seed {seed} (bucket {data.bucket}): {len(RUNG_SPECS) * 2} fits in "
          f"{time.perf_counter() - t0:.1f}s  "
          f"[{len(data.split.train):,} train / {len(data.split.test):,} test rows]")
    del data
    gc.collect()

realised = pd.DataFrame(realised_rows, columns=["seed", "bucket", "n_train", "n_test"])
print()
print("realised bucket sizes (a property of the data, not a number chosen here):")
print(realised.to_string(index=False))

print(f"\n{len(all_results)} results total "
      f"({len(RUNG_SPECS)} rungs x 2 variants x {len(REPEAT_SEEDS)} seeds, plus references)")

seed 1 (bucket 0): 16 fits in 441.5s  [1,955,364 train / 420,169 test rows]


seed 2 (bucket 1): 16 fits in 420.6s  [1,957,342 train / 420,267 test rows]


seed 3 (bucket 2): 16 fits in 419.4s  [1,955,147 train / 421,283 test rows]


seed 4 (bucket 3): 16 fits in 416.8s  [1,959,559 train / 421,535 test rows]


seed 5 (bucket 4): 16 fits in 498.3s  [1,957,859 train / 421,747 test rows]

realised bucket sizes (a property of the data, not a number chosen here):
 seed  bucket  n_train  n_test
    1       0  1955364  420169
    2       1  1957342  420267
    3       2  1955147  421283
    4       3  1959559  421535
    5       4  1957859  421747

90 results total (8 rungs x 2 variants x 5 seeds, plus references)


## 9. Results with uncertainty attached

`mae_seconds` / `mape_pct` keep their original meaning — the **first repeat**
— so the columns written before this pass still say something concrete. Every
claim in `docs/modelling_notes.md` is now made against `mae_mean` and
qualified by `mae_sd`.

Read `mae_sd` first: it is how much a rung's error moves when nothing
meaningful changes, and any rung-to-rung gap of comparable size is noise.

In [17]:
table_b = aggregate_repeats(all_results, mae_col="mae_seconds")

settings.TABLES_DIR.mkdir(parents=True, exist_ok=True)
table_b_path = settings.TABLES_DIR / "model_ladder_nyc.csv"
table_b.to_csv(table_b_path, index=False)
print(f"wrote {table_b_path}  ({len(table_b)} rows)")
table_b[["variant", "rung", "mae_seconds", "mae_mean", "mae_sd", "mae_min", "mae_max",
         "mape_mean", "mape_sd", "n_repeats"]]

wrote /app/outputs/tables/model_ladder_nyc.csv  (18 rows)


,variant,rung,mae_seconds,mae_mean,mae_sd,mae_min,mae_max,mape_mean,mape_sd,n_repeats
0,reference (no model fitted),REF_mean,647.3819,646.9630,3.0473,641.7781,649.8334,210.0233,2.2737,5
1,reference (no model fitted),REF_heuristic,549.6182,548.6071,2.6500,544.3228,551.5260,46.7433,1.1723,5
2,"original (untuned, 300 trees)",V0,336.8535,337.6391,2.0830,334.5269,339.5860,78.0921,1.7086,5
3,capacity-controlled (CV-selected on V0),V0,336.4698,337.3219,2.1902,334.1656,339.7582,77.5639,2.4170,5
4,"original (untuned, 300 trees)",V1,336.1267,336.3589,2.2022,333.0533,339.2026,74.5251,2.4966,5
5,capacity-controlled (CV-selected on V0),V1,335.5913,335.9910,2.2593,332.6085,338.8797,73.5801,2.4833,5
6,"original (untuned, 300 trees)",V2,330.7424,330.4769,2.2656,326.7962,333.0111,72.3969,2.1335,5
7,capacity-controlled (CV-selected on V0),V2,330.9188,330.7674,2.2610,327.0927,333.2534,72.0043,2.4653,5
8,"original (untuned, 300 trees)",V3a,326.2630,325.7728,2.2277,322.1835,328.3319,70.7649,1.7869,5
9,capacity-controlled (CV-selected on V0),V3a,326.2250,325.8635,2.2362,322.2631,328.4504,70.3233,2.0462,5


## 10. Paired comparisons: which differences survive repetition?

Differences are taken **within seed** — the same bucket and the same model
randomness for both rungs — then summarised across the five repeats.
`mean_diff_mae` is rung B minus rung A: **negative means B is better.**
`n_same_direction` counts how many repeats agreed with the mean's sign.

Nothing here is a significance test and none is claimed: with five repeats
these are descriptive indications of stability. A difference that does not
hold its sign in all five repeats is not an established effect.

In [18]:
RUNG_ORDER = [s["rung"] for s in RUNG_SPECS]

best_original = best_rung_by_mean_mae(table_b, VARIANT_ORIGINAL)
best_capacity = best_rung_by_mean_mae(table_b, VARIANT_CAPACITY_CONTROLLED)
print(f"best rung by mean MAE -- original: {best_original}, capacity-controlled: {best_capacity}")
print("(V3a and the REF_* rows are excluded from 'best': leaky, and not rungs)")

special = [
    Pair("V3a", "V3b", label="leaky vs point-in-time correct"),
    Pair("V4a", "V4b", label="one-hot vs point-in-time target encoding"),
]
for variant, best in ((VARIANT_ORIGINAL, best_original),
                      (VARIANT_CAPACITY_CONTROLLED, best_capacity)):
    if best == "V0":
        print(f"  [{variant}] the best rung IS V0: no engineered rung beat the raw baseline.")
    elif not any(p.rung_a == "V0" and p.rung_b == best for p in special):
        special.append(Pair("V0", best, label="V0 vs best rung"))

seen = {(p.rung_a, p.rung_b) for p in special}
pairs = special + [p for p in adjacent_pairs(RUNG_ORDER) if (p.rung_a, p.rung_b) not in seen]

paired_b = paired_comparisons(all_results, pairs, task="nyc", mae_col="mae_seconds")
paired_b_path = settings.TABLES_DIR / "ladder_paired_comparisons_nyc.csv"
paired_b.to_csv(paired_b_path, index=False)
print(f"wrote {paired_b_path}  ({len(paired_b)} comparisons)")

paired_b[["variant", "rung_a", "rung_b", "comparison", "mean_diff_mae", "sd_diff_mae",
          "mean_over_sd", "n_same_direction", "direction_consistent", "better_rung"]]

best rung by mean MAE -- original: V4a, capacity-controlled: V4a
(V3a and the REF_* rows are excluded from 'best': leaky, and not rungs)
wrote /app/outputs/tables/ladder_paired_comparisons_nyc.csv  (16 comparisons)


,variant,rung_a,rung_b,comparison,mean_diff_mae,sd_diff_mae,mean_over_sd,n_same_direction,direction_consistent,better_rung
0,"original (untuned, 300 trees)",V3a,V3b,leaky vs point-in-time correct,2.0246,0.2578,7.853,5,True,V3a
1,"original (untuned, 300 trees)",V4a,V4b,one-hot vs point-in-time target encoding,4.5610,1.9050,2.394,5,True,V4a
2,"original (untuned, 300 trees)",V0,V4a,V0 vs best rung,-13.5343,0.8193,-16.520,5,True,V4a
3,"original (untuned, 300 trees)",V0,V1,adjacent,-1.2802,0.8193,-1.562,5,True,V1
4,"original (untuned, 300 trees)",V1,V2,adjacent,-5.8819,0.3543,-16.602,5,True,V2
5,"original (untuned, 300 trees)",V2,V3a,adjacent,-4.7042,0.1856,-25.340,5,True,V3a
6,"original (untuned, 300 trees)",V3b,V4a,adjacent,-3.6926,0.3144,-11.744,5,True,V4a
7,"original (untuned, 300 trees)",V4b,V5,adjacent,3.2919,1.1953,2.754,5,True,V4b
8,capacity-controlled (CV-selected on V0),V3a,V3b,leaky vs point-in-time correct,2.2368,0.3401,6.578,5,True,V3a
9,capacity-controlled (CV-selected on V0),V4a,V4b,one-hot vs point-in-time target encoding,5.9035,2.5548,2.311,5,True,V4a


### Do the rungs beat the reference predictors at all?

In [19]:
ref_rows = table_b[table_b["variant"] == VARIANT_REFERENCE].set_index("rung")
ref_speed_mae = float(ref_rows.loc["REF_heuristic", "mae_mean"])
ref_mean_mae = float(ref_rows.loc["REF_mean", "mae_mean"])
print(f"REF_heuristic (distance / mean speed): MAE={ref_speed_mae:,.1f}s "
      f"(sd {float(ref_rows.loc['REF_heuristic', 'mae_sd']):.1f})")
print(f"REF_mean (training mean):              MAE={ref_mean_mae:,.1f}s "
      f"(sd {float(ref_rows.loc['REF_mean', 'mae_sd']):.1f})")
print()

long_b = results_to_long_frame(all_results, mae_col="mae_seconds")
fitted_long = long_b[long_b["variant"] != VARIANT_REFERENCE]
for variant in (VARIANT_ORIGINAL, VARIANT_CAPACITY_CONTROLLED):
    print(f"[{variant}]")
    scoped = fitted_long[fitted_long["variant"] == variant]
    for rung in RUNG_ORDER:
        maes = scoped[scoped["rung"] == rung]["mae_seconds"]
        print(f"  {rung:4s} beats distance/speed in {int((maes < ref_speed_mae).sum())}/{len(maes)}"
              f" repeats;  beats training mean in {int((maes < ref_mean_mae).sum())}/{len(maes)}"
              " repeats")
    print()

REF_heuristic (distance / mean speed): MAE=548.6s (sd 2.6)
REF_mean (training mean):              MAE=647.0s (sd 3.0)

[original (untuned, 300 trees)]
  V0   beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V1   beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V2   beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V3a  beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V3b  beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V4a  beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V4b  beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V5   beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats

[capacity-controlled (CV-selected on V0)]
  V0   beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V1   beats distance/speed in 5/5 repeats;  beats training mean in 5/5 repeats
  V2  

In [20]:
inconsistent = paired_b[~paired_b["direction_consistent"]]
consistent = paired_b[paired_b["direction_consistent"]]
print(f"{len(consistent)} of {len(paired_b)} comparisons held their direction across all "
      f"{len(REPEAT_SEEDS)} repeats.")
print()
if len(inconsistent):
    print("NOT consistent in direction -- these must not be reported as established effects:")
    for _, row in inconsistent.iterrows():
        print(f"  [{row['variant'][:22]:22s}] {row['rung_a']:>4s} -> {row['rung_b']:<4s} "
              f"mean {row['mean_diff_mae']:+9.2f}s  sd {row['sd_diff_mae']:8.2f}  "
              f"{row['n_same_direction']}/{row['n_repeats']} agree")
else:
    print("every comparison held its direction across all repeats.")

15 of 16 comparisons held their direction across all 5 repeats.

NOT consistent in direction -- these must not be reported as established effects:
  [capacity-controlled (C]  V4b -> V5   mean     +3.13s  sd     2.65  4/5 agree


## 11. The figure: rungs with error bars, against the reference predictors

In [21]:
vizstyle.apply_style()

summary = table_b.set_index(["variant", "rung"])
rungs = RUNG_ORDER
mape_o = [summary.loc[(VARIANT_ORIGINAL, r), "mape_mean"] for r in rungs]
sd_o = [summary.loc[(VARIANT_ORIGINAL, r), "mape_sd"] for r in rungs]
mape_c = [summary.loc[(VARIANT_CAPACITY_CONTROLLED, r), "mape_mean"] for r in rungs]
sd_c = [summary.loc[(VARIANT_CAPACITY_CONTROLLED, r), "mape_sd"] for r in rungs]
ref_mean_mape = summary.loc[(VARIANT_REFERENCE, "REF_mean"), "mape_mean"]
ref_speed_mape = summary.loc[(VARIANT_REFERENCE, "REF_heuristic"), "mape_mean"]

# REF_mean sits at ~210% MAPE while every rung sits at 70-78%, so a single
# axis compresses the rungs -- and their error bars, the point of the figure --
# into an unreadable band. A broken axis keeps BOTH reference lines on the
# chart (the distance/speed heuristic falls naturally beside the rungs) while
# giving the rungs enough vertical room to be compared.
fig, (ax_top, ax) = plt.subplots(
    2, 1, sharex=True, figsize=(9.0, 6.4),
    gridspec_kw={"height_ratios": [1, 5], "hspace": 0.08},
)

x = np.arange(len(rungs))
width = 0.38
colours_o = [vizstyle.PALETTE[0]] * len(rungs)
colours_c = [vizstyle.PALETTE[2]] * len(rungs)
hatches = ["////" if r == "V3a" else None for r in rungs]

for axis in (ax_top, ax):
    axis.bar(x - width / 2, mape_o, width=width, yerr=sd_o, capsize=3,
             color=colours_o, hatch=hatches, edgecolor="white", linewidth=0.6,
             error_kw={"ecolor": vizstyle.TEXT_COLOUR, "elinewidth": 1.0},
             label="original (untuned, 300 trees)")
    axis.bar(x + width / 2, mape_c, width=width, yerr=sd_c, capsize=3,
             color=colours_c, hatch=hatches, edgecolor="white", linewidth=0.6,
             error_kw={"ecolor": vizstyle.TEXT_COLOUR, "elinewidth": 1.0},
             label="capacity-controlled (CV-selected)")
    axis.axhline(ref_speed_mape, color=vizstyle.PALETTE[3], linestyle="--", linewidth=1.4,
                 label=f"REF distance/speed ({ref_speed_mape:.1f}%)")
    axis.axhline(ref_mean_mape, color=vizstyle.MUTED_COLOUR, linestyle=":", linewidth=1.4,
                 label=f"REF training mean ({ref_mean_mape:.1f}%)")

# Top: only the far-off training-mean reference. Bottom: the rungs and the
# heuristic, on a scale where a one-point difference is actually visible.
ax_top.set_ylim(ref_mean_mape - 6, ref_mean_mape + 6)
rung_lo = min(min(mape_o), min(mape_c), ref_speed_mape) - 6
rung_hi = max(max(m + s for m, s in zip(mape_o, sd_o)),
              max(m + s for m, s in zip(mape_c, sd_c))) + 6
ax.set_ylim(rung_lo, rung_hi)

# Hide the facing spines and draw the conventional break marks.
ax_top.spines["bottom"].set_visible(False)
ax.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", which="both", bottom=False)
ax_top.tick_params(axis="y", labelsize=9)
kw = dict(marker=[(-1, -0.6), (1, 0.6)], markersize=7, linestyle="none",
          color=vizstyle.TEXT_COLOUR, mec=vizstyle.TEXT_COLOUR, mew=1, clip_on=False)
ax_top.plot([0, 1], [0, 0], transform=ax_top.transAxes, **kw)
ax.plot([0, 1], [1, 1], transform=ax.transAxes, **kw)

ax.set_ylabel("MAPE on held-out test set (%)", fontsize=10)
ax.yaxis.set_label_coords(-0.075, 0.62)
ax.set_xlabel("Ladder rung", fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(rungs, fontsize=10)
ax.tick_params(axis="y", labelsize=9)
ax_top.set_title("NYC trip duration, pickup-time features: MAPE by rung, with repeat spread",
                 fontsize=11, fontweight="bold")
ax.legend(fontsize=8.5, loc="upper left", ncol=2)
fig.tight_layout()

fig_path = vizstyle.finish(
    fig, settings.FIGURES_DIR / "fig_model_ladder_nyc.png",
    "Source: this platform's own gold layer (fact_trip, fact_weather_daily) "
    "and the raw NYC TLC 2024 corpus (V0 only)",
    f"Bars are the mean of {len(REPEAT_SEEDS)} repeats (seeds {REPEAT_SEEDS[0]}-{REPEAT_SEEDS[-1]}, "
    f"each drawing its own deterministic content-addressed bucket of about "
    f"{realised['n_train'].mean():,.0f} training rows and using its own model random_state); "
    f"error bars are one standard "
    "deviation across those repeats, i.e. the amount a rung moves when nothing meaningful "
    "changes. The y-axis is broken and does not start at zero: the training-mean reference "
    "sits far above every rung, and on a single zero-based axis the rungs and their error "
    "bars compress into an unreadable band. Dashed and dotted lines are reference predictors "
    "that fit no model: distance over "
    "average speed, and the training mean. V3a (hatched) is computed with future months present "
    "and is reported only for comparison against V3b. Which rung-to-rung differences hold their "
    "direction across all repeats is in outputs/tables/ladder_paired_comparisons.csv, and is not "
    "readable from bar heights alone.",
)
print(f"wrote {fig_path}")

2026-09-19 23:26:07 | INFO    | src.viz.style                | figure written: fig_model_ladder_nyc.png


wrote /app/outputs/figures/fig_model_ladder_nyc.png


## 12. Notebook runtime

In [22]:
elapsed = time.perf_counter() - NOTEBOOK_STARTED
print(f"notebook 02 wall-clock time: {elapsed:.1f}s ({elapsed/60:.2f} min)")
print(f"  {len(RUNG_SPECS)} rungs x 2 variants x {len(REPEAT_SEEDS)} repeats = "
      f"{len(RUNG_SPECS) * 2 * len(REPEAT_SEEDS)} fits, plus a {tuning.seconds:.1f}s CV search "
      f"and {len(REPEAT_SEEDS)} independent data loads")

notebook 02 wall-clock time: 3803.1s (63.39 min)
  8 rungs x 2 variants x 5 repeats = 80 fits, plus a 950.8s CV search and 5 independent data loads
